In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

cols = ['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty']
df_train = pd.read_csv('data/KDDTrain+.txt', header=None, names=cols)
df_test = pd.read_csv('data/KDDTest+.txt', header=None, names=cols)
print(df_train.shape, df_test.shape)


(125973, 43) (22544, 43)


In [11]:
y_train = (df_train['label'] != 'normal').astype(int)
y_test = (df_test['label'] != 'normal').astype(int)


In [12]:
df_all = pd.concat([df_train, df_test], axis=0)
df_all = pd.get_dummies(df_all, columns=['protocol_type','service','flag'])
df_train = df_all.iloc[:125973].copy()
df_test = df_all.iloc[125973:].copy()


In [13]:
X_train = df_train.drop(['label','difficulty'], axis=1)
X_test = df_test.drop(['label','difficulty'], axis=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(X_train_scaled.shape, X_test_scaled.shape)


(125973, 122) (22544, 122)


In [14]:
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train)
print(X_tr.shape, X_val.shape)


(100778, 122) (25195, 122)


In [15]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
y_pred = model.predict(X_val)


In [16]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("准确率:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred, target_names=['正常', '攻击']))
print(confusion_matrix(y_val, y_pred))


准确率: 0.9723357809089105
              precision    recall  f1-score   support

          正常       0.97      0.98      0.97     13469
          攻击       0.98      0.96      0.97     11726

    accuracy                           0.97     25195
   macro avg       0.97      0.97      0.97     25195
weighted avg       0.97      0.97      0.97     25195

[[13198   271]
 [  426 11300]]
